<b><font size="6" color="#E8800A">Week 9a · Decision trees</font></b><br>
<b><font size="4">Growing a tree, pruning it, and following one prediction</font></b><br>

A decision tree answers a question about an athlete with a path of yes-or-no questions, each comparing one column with one threshold. The path ends in a leaf, and the training athletes that landed in that leaf decide the answer. How many questions the tree may ask, and how large a leaf must be, set how closely it follows the training athletes.

<div class="alert alert-block alert-info">

# TOC<a class="anchor" id="toc"></a>
* [<font color='#E8800A'>How a tree chooses a split</font>](#split)
* [<font color='#E8800A'>Fitting a tree</font>](#fit)
* [<font color='#E8800A'>How a tree predicts</font>](#predict)
* [<font color='#E8800A'>What a tree cannot do</font>](#limits)
* [<font color='#E8800A'>The settings of a tree</font>](#settings)
* [<font color='#E8800A'>Two ways to prune</font>](#prune)
* [<font color='#E8800A'>Nested cross-validation</font>](#nested)
* [<font color='#E8800A'>The grid search</font>](#search)
* [<font color='#E8800A'>The number we stand behind</font>](#board)
* [<font color='#E8800A'>Key takeaways</font>](#takeaways)
* [<font color='#E8800A'>References</font>](#references)

</div>

# <font color='#E8800A'>How a tree chooses a split</font> <a class="anchor" id="split"></a>
[Back to TOC](#toc)

A tree is built one question at a time. Each question splits the athletes in a node into two groups, chosen so that the outcomes in each group are less mixed than in the node. A toy example of eight athletes, small enough to compute every quantity, shows how the first question is chosen, and a fitted tree is drawn beside it.

In [ ]:
from tqdm.auto import tqdm

%pip install -q --disable-pip-version-check supertree==1.1.0

import sys
from pathlib import Path

# course_helpers.py and preprocessing.py sit in this folder, next to the notebook.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import ParameterGrid, cross_validate, train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from supertree import SuperTree

from course_helpers import (
    PLOT_BLUE,
    PLOT_ORANGE,
    Scoreboard,
    record,
    result_context,
    results_before,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("svg")
random_state = 33
np.random.seed(random_state)  # For reproducibility

# Fold-safe encoding means a rare level can occur only in validation, so this
# message would repeat once per scored fold.
warnings.filterwarnings(
    "ignore",
    message="Found unknown categories.*during transform",
    category=UserWarning,
)

from preprocessing import (
    CachedSearchCV,
    FoldPreprocessor,
    PreparedEstimator,
    load_recipe,
    outer_folds,
    outer_splitter,
    prepared_folds,
)
classification_recipe_log = "../../logs/week_04_feature_work_classification_log.json"


__Step 1:__ Write down eight toy athletes: weekly training hours, hours of sleep, and whether the athlete won.

In [ ]:
toy = pd.DataFrame([[4, 8, 0], [6, 7, 0], [8, 6, 0], [10, 7, 1], [12, 8, 1], [14, 5, 0], [16, 7, 1], [18, 8, 1]],
                   columns=["hours", "sleep", "won"])
toy.T

The **Gini impurity** of a group measures how mixed the outcomes of its athletes are:

$$G = 1 - \sum_{c=0}^{1} p_c^{2},$$

where $c$ runs over the two outcomes, 0 for lost and 1 for won, and $p_c$ is the share of the group's athletes with outcome $c$. A group whose athletes all share one outcome has $G = 0$, and equal shares give the largest value, $G = 0.5$.

__Step 2:__ The Gini impurity of the eight athletes: the share of each outcome, then the impurity. Type the impurity as `gini_parent = 1 - (shares ** 2).sum()`.

In [ ]:
shares = toy["won"].value_counts(normalize=True)
gini_parent = ...
print(f"shares of the outcomes: {shares.round(4).to_dict()}")
print(f"Gini impurity of the eight athletes: {gini_parent:.4f}")

A split sends the athletes whose value in one column is at most a threshold to the left child $L$, and the others to the right child $R$. Its **impurity decrease** is

$$\Delta = G_P - \frac{n_L}{n_P}\,G_L - \frac{n_R}{n_P}\,G_R,$$

where $P$ is the group being split, $n_P$, $n_L$ and $n_R$ are the numbers of athletes in the group and in its two children, and $G_P$, $G_L$ and $G_R$ are their Gini impurities. The weights $n_L / n_P$ and $n_R / n_P$ add to 1, so the last two terms are the children's impurity averaged by size. The tree tries every threshold of every column, taking the midpoint between neighbouring values, and keeps the split with the largest $\Delta$.

__Step 3:__ Every candidate threshold of both columns: the athletes each side receives, the impurity of each side and the impurity decrease. Type the two side impurities as `gini_left = 1 - (left.value_counts(normalize=True) ** 2).sum()` and `gini_right = 1 - (right.value_counts(normalize=True) ** 2).sum()`.

In [ ]:
candidates = []
for column in ["hours", "sleep"]:
    values = np.sort(toy[column].unique())
    for threshold in (values[:-1] + values[1:]) / 2:
        left, right = toy["won"][toy[column] <= threshold], toy["won"][toy[column] > threshold]
        gini_left = ...
        gini_right = ...
        weighted = (len(left) * gini_left + len(right) * gini_right) / len(toy)
        candidates.append({"column": column, "threshold": threshold, "left": len(left), "right": len(right),
                           "gini_left": gini_left, "gini_right": gini_right, "weighted": weighted,
                           "decrease": gini_parent - weighted})
candidates = pd.DataFrame(candidates).sort_values("decrease", ascending=False)
candidates.head(5).round(4)

**The best candidate is `hours <= 9.0`, with an impurity decrease of 0.3000: it sends 3 athletes left, all of whom lost, at a Gini impurity of 0.0000, and 5 right at 0.3200, which weight to 0.2000, against the parent's 0.5000.** Three other candidates, `hours <= 7.0`, `sleep <= 6.5` and `hours <= 15.0`, tie at 0.1667.

__Step 4:__ Fit a tree limited to two questions on the same eight athletes: `DecisionTreeClassifier(max_depth=2, random_state=random_state)`.

In [ ]:
toy_tree = ...
toy_tree = toy_tree.fit(toy[["hours", "sleep"]], toy["won"])

__Step 5:__ Draw the toy tree.

In [ ]:
figure, axis = plt.subplots(figsize=(10, 5))
plot_tree(toy_tree, feature_names=["hours", "sleep"], class_names=["lost", "won"], filled=True,
          rounded=True, precision=2, fontsize=11, ax=axis)
plt.show()

**How to read a box.** A box that has children starts with its question: athletes whose value is at most the threshold go left, the others go right, and a leaf has no question. `gini` is the Gini impurity of the athletes that reach the box, `samples` is how many of the eight reach it, and `value` counts them by outcome, lost first and won second. `class` is the outcome with the larger count, which is the answer a leaf gives, and lost when the counts are equal. The colour follows `class`, orange for lost and blue for won, and is deeper when the box is purer, so a box with equal counts is white. The root reads `hours <= 9.0`, `gini = 0.5`, `samples = 8` and `value = [4, 4]`, and its left child holds the 3 athletes who lost, with `gini = 0.0` and `value = [3, 0]`.

__Step 6:__ The first question of the fitted tree, beside the best candidate of the table.

In [ ]:
best_candidate = candidates.iloc[0]
print(f"by hand: {best_candidate['column']} <= {best_candidate['threshold']}, impurity decrease {best_candidate['decrease']:.4f}")
print(f"fitted:  {['hours', 'sleep'][toy_tree.tree_.feature[0]]} <= {toy_tree.tree_.threshold[0]},"
      f" impurity {toy_tree.tree_.impurity[0]:.4f}, then {toy_tree.tree_.impurity[toy_tree.tree_.children_left[0]]:.4f}"
      f" and {toy_tree.tree_.impurity[toy_tree.tree_.children_right[0]]:.4f}")

**The fitted tree asks `hours <= 9.0` first, the table's best candidate, and its impurities, 0.5000, 0.0000 and 0.3200, are the table's.** The five athletes on the right ask a second question about `sleep`, which separates the one athlete who lost from the four who won.

__Step 7:__ Fit a tree with one question on the cubes of the two columns, and compare its groups with the first question's.

In [ ]:
cubed_tree = DecisionTreeClassifier(max_depth=1, random_state=random_state).fit(toy[["hours", "sleep"]] ** 3, toy["won"])
plain_tree = DecisionTreeClassifier(max_depth=1, random_state=random_state).fit(toy[["hours", "sleep"]], toy["won"])
print(f"cubed question: {['hours', 'sleep'][cubed_tree.tree_.feature[0]]} <= {cubed_tree.tree_.threshold[0]}, in cubed units")
print(f"same groups of athletes: {np.array_equal(plain_tree.apply(toy[['hours', 'sleep']]), cubed_tree.apply(toy[['hours', 'sleep']] ** 3))}")

**The cubed columns give the same groups.** The question moves to `hours <= 756.0` in cubed units: 756.0 is the midpoint of the cubes 512 and 1000, not the cube of 9.0, and the groups of athletes are the same. A tree reads the order of a column's values, so a strictly increasing change of a column, such as the log or the scaling of the recipe, leaves the training groups unchanged. The threshold is the midpoint of two neighbouring training values, and a nonlinear change moves that midpoint, so a new value between those two could change sides.

# <font color='#E8800A'>Fitting a tree</font> <a class="anchor" id="fit"></a>
[Back to TOC](#toc)

Fitting grows the tree on the prepared training athletes: each node takes the split with the largest impurity reduction, and stops splitting when no column can separate its athletes or a setting forbids another split. The default tree has no setting that forbids one.

__Step 8:__ Load the classification spine and split it.

In [ ]:
data, classification_recipe = load_recipe(
    "../../data/interim/champions.csv",
    classification_recipe_log,
)
# RecordID names each row, so it becomes the index.
data = data.set_index("RecordID")
features = data.loc[:, classification_recipe["categorical"] + classification_recipe["numeric"]].copy()
target = data["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, stratify=target, random_state=random_state,
)
print(f"{len(X_train):,} training athletes, {len(X_test):,} test athletes,"
      f" positive rate {y_train.mean():.4f}")

__Step 9:__ Prepare the athletes, and fix the folds every score is read on.

In [ ]:
tree_prep = FoldPreprocessor(classification_recipe).fit(X_train, y_train)
Z_train = tree_prep.transform(X_train)
Z_test = tree_prep.transform(X_test)
prepared_names = list(tree_prep.get_feature_names_out())

# The ten folds every validation score is read on.
tree_folds = outer_splitter("classification", random_state=random_state)
tree_prepared = prepared_folds(
    FoldPreprocessor(classification_recipe), tree_folds, X_train, y_train, n_jobs=4)
print(f"{Z_train.shape[1]} prepared columns, {tree_folds.get_n_splits()} folds")

<div class="alert alert-block alert-info">
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html'>sklearn.tree.DecisionTreeClassifier(*, criterion='gini', splitter='best', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=None, random_state=None, max_leaf_nodes=None, min_impurity_decrease=0.0, class_weight=None, ccp_alpha=0.0, monotonic_cst=None)</a>

**Definition:**
A binary tree that classifies a row by the class shares of the training rows in its leaf.

**Main Parameters:**
- `max_depth` (default `None`): the most questions on any path.
- `min_samples_leaf` (default `1`): the fewest training rows a leaf may hold.
- `min_samples_split` (default `2`): the fewest training rows a node needs to ask another question.
- `criterion` (default `'gini'`): the impurity a split must lower; `'entropy'` and `'log_loss'` are the same alternative.
- `max_features` (default `None`): how many columns each split considers.
- `ccp_alpha` (default `0.0`): the cost of a leaf in cost-complexity pruning.
- `class_weight` (default `None`): the weight of each outcome; `'balanced'` weights them inversely to their frequency.
- `random_state`: fixes the order in which columns are tried, which decides ties.

**Main Attributes (after fitting):**
- `tree_`: the fitted tree: for each node its children, column, threshold, training rows and impurity.
- `classes_`: the class labels, in the order `predict_proba` uses.
- `feature_importances_`: each column's total impurity reduction, scaled to sum to 1.

**Main Methods:**
- `fit(X, y)`: grows the tree.
- `apply(X)`: the leaf each row reaches.
- `decision_path(X)`: the nodes each row passes through.
- `predict_proba(X)` and `predict(X)`: the class shares of the leaf, and the larger one.
- `get_depth()` and `get_n_leaves()`: the size of the fitted tree.
- `cost_complexity_pruning_path(X, y)`: the alphas at which branches would be pruned.
</div>

__Step 10:__ Score `DecisionTreeClassifier()` on the ten folds, and open the log.

In [ ]:
tree_log = []
default_model = DecisionTreeClassifier(random_state=random_state)
default_trees = [clone(default_model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(tree_prepared, desc='tree prepared', leave=True, position=0)]
default_cv = {"train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                        for fold, tree in zip(tree_prepared, default_trees)],
              "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                             for fold, tree in zip(tree_prepared, default_trees)]}
default_fitted = clone(default_model).fit(Z_train, y_train)

In [ ]:
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": "DecisionTreeClassifier()",
                  "train": default_cv["train"], "validation": default_cv["validation"],
                  "model": default_fitted},
       Z_test, y_test, metric=f1_score, label="F1")

**The default tree scores F1 1.0000 on the folds it was fitted on, 0.7996 on the validation folds and 0.8041 on the test rows.** It keeps asking questions until its leaves cannot be split, so it reproduces every training athlete, and the gap between the first two scores, 0.2004, is how far it follows the training athletes beyond what carries to new ones.

__Step 11:__ Draw the whole default tree, every node, in a zoomable viewer whose footer states its nodes, leaves and depth.

In [ ]:
SuperTree(default_fitted, Z_train, y_train.to_numpy(), prepared_names, ["lost", "won"]).show_tree(start_depth=default_fitted.get_depth(), max_samples=500)

**Drawn in full, the default tree has 21 levels and 504 leaves, too many boxes to read.** Its training F1 of 1.0000 means this whole structure answers the training athletes without error. In the viewer, each leaf is a pie of its athletes with the counts of lost and won, readable once zoomed in.

__Step 12:__ Score the tree limited to two questions on the same folds, and print its scores beside the default's.

In [ ]:
small_model = DecisionTreeClassifier(max_depth=2, random_state=random_state)
small_trees = [clone(small_model).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(tree_prepared, desc='tree prepared', leave=True, position=0)]
small_cv = {"train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                      for fold, tree in zip(tree_prepared, small_trees)],
            "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                           for fold, tree in zip(tree_prepared, small_trees)]}
small_tree = clone(small_model).fit(Z_train, y_train)

In [ ]:
print(f"depth two: training F1 {np.mean(small_cv['train']):.4f}, validation F1 {np.mean(small_cv['validation']):.4f}")
print(f"default:   training F1 {np.mean(default_cv['train']):.4f}, validation F1 {np.mean(default_cv['validation']):.4f}")

__Step 13:__ Read the depth and the number of leaves of the two fitted trees.

In [ ]:
pd.DataFrame([{"tree": "default", "depth": default_fitted.get_depth(), "leaves": default_fitted.get_n_leaves()},
              {"tree": "depth two", "depth": small_tree.get_depth(), "leaves": small_tree.get_n_leaves()}]).set_index("tree")

**The default tree is 21 questions deep with 504 leaves, and the depth-two tree has 4 leaves.** The depth-two tree scores 0.7968 on its training folds and 0.7923 on its validation folds: the two are close, and the validation score is below the default's 0.7996, so two questions are too few.

# <font color='#E8800A'>How a tree predicts</font> <a class="anchor" id="predict"></a>
[Back to TOC](#toc)

A fitted tree is a set of nested questions, so it can be drawn in full. The depth-two tree below has four leaves, and the steps that follow take the first test athlete from the top question to its leaf and to its prediction.

__Step 14:__ Draw the depth-two tree fitted on all the training rows, in the zoomable viewer.

In [ ]:
SuperTree(small_tree, Z_train, y_train.to_numpy(), prepared_names, ["lost", "won"]).show_tree(start_depth=small_tree.get_depth(), max_samples=500)

**The top question is on `Train bf competition` at -0.299, and its two children ask about `Cardiovascular training` at -0.197 and `Train bf competition` at 0.342.** Each question box draws a histogram of its column over the athletes that reach it, split by outcome, with a dashed line at the threshold, and each leaf is a pie of the outcomes with their counts. The four leaves hold 487 lost and 82 won, 291 lost and 247 won, 353 lost and 741 won, and 136 lost and 863 won. The second leaf still answers lost with 247 athletes who won: a leaf answers with the larger count however close the two are.

__Step 15:__ Follow the first test athlete through the tree: at each question, its prepared value, the threshold and the branch taken.

In [ ]:
example = Z_test[:1]
trace = []
for node in small_tree.decision_path(example).indices:
    column = small_tree.tree_.feature[node]
    if column >= 0:
        trace.append({"node": node, "column": prepared_names[column], "value": example[0, column],
                      "threshold": small_tree.tree_.threshold[node],
                      "branch": "left" if example[0, column] <= small_tree.tree_.threshold[node] else "right"})
leaf = small_tree.apply(example)[0]
print(pd.DataFrame(trace).round(4).to_string(index=False))
print(f"leaf {leaf}, which holds {small_tree.tree_.n_node_samples[leaf]} training athletes")

__Step 16:__ The leaf's training athletes by outcome, and the share of each: the probability by hand.

In [ ]:
in_leaf = small_tree.apply(Z_train) == leaf
counts = np.bincount(y_train.to_numpy()[in_leaf], minlength=2)
print(pd.DataFrame({"outcome": [0, 1], "training athletes": counts, "share": counts / counts.sum()}).round(4).to_string(index=False))

__Step 17:__ `predict_proba`: the same shares from the fitted tree, then `predict` beside the athlete's outcome.

In [ ]:
print(f"predict_proba: {small_tree.predict_proba(example)[0].round(4)}")
print(f"predict: {small_tree.predict(example)[0]}; the athlete's outcome: {y_test.iloc[0]}")

**The first test athlete reaches the right-hand leaf that holds 1,094 training athletes, after two comparisons of `Train bf competition`.** Its prepared value, -0.0466, is above -0.299, so it goes right, and below 0.342, so it goes left. Of the 1,094 athletes in that leaf 741 won and 353 lost, so the probability of winning is 0.6773 by hand and from `predict_proba`, and `predict` returns 1, the athlete's outcome.

# <font color='#E8800A'>What a tree cannot do</font> <a class="anchor" id="limits"></a>
[Back to TOC](#toc)

A leaf answers every athlete that reaches it in the same way. Three consequences are read from the default tree: how few different answers it can give, what it does with an athlete unlike any it was fitted on, and what its column importances measure.

__Step 18:__ Count the different probabilities each tree gives the test athletes, and the share that is exactly 0 or 1.

In [ ]:
default_probability = default_fitted.predict_proba(Z_test)[:, 1]
small_probability = small_tree.predict_proba(Z_test)[:, 1]
print(f"default: {len(np.unique(default_probability))} different probabilities;"
      f" {np.mean((default_probability == 0) | (default_probability == 1)):.4f} of the test athletes get exactly 0 or 1")
print(f"depth two: {len(np.unique(small_probability))} different probabilities;"
      f" {np.mean((small_probability == 0) | (small_probability == 1)):.4f} of the test athletes get exactly 0 or 1")

__Step 19:__ Count the training athletes in each leaf of the default tree.

In [ ]:
leaf_sizes = default_fitted.tree_.n_node_samples[default_fitted.tree_.children_left == -1]
print(f"{len(leaf_sizes)} leaves; median {np.median(leaf_sizes):.0f} training athletes per leaf;"
      f" {np.mean(leaf_sizes == 1):.4f} of the leaves hold a single athlete")

**The default tree gives the test athletes 2 different probabilities, 0 and 1, and the depth-two tree gives 4, none of them 0 or 1.** The default's 504 leaves hold a median of 2 training athletes, and 0.3790 of them hold a single athlete, so a probability of 0 or 1 from this tree reports one or two athletes and not certainty.

__Step 20:__ Build a probe unlike any training athlete: the first one, with every numeric column set to ten times its largest training value.

In [ ]:
probe = X_train.iloc[[0]].copy()
numeric = classification_recipe["numeric"]
probe.loc[:, numeric] = 10 * X_train[numeric].max().to_numpy()
probe_ready = tree_prep.transform(probe)

__Step 21:__ Ask the default tree about the probe: its probability of winning, and the size of the leaf that answers.

In [ ]:
print(f"probability of winning: {default_fitted.predict_proba(probe_ready)[0, 1]:.4f}")
print(f"training athletes in its leaf: {default_fitted.tree_.n_node_samples[default_fitted.apply(probe_ready)[0]]}")

**The probe, with every numeric column ten times its largest training value, still receives an answer: a probability of winning of 0.0000, from a leaf of 2 training athletes.** The tree compared the probe's values with its thresholds as it would for any athlete, and nothing in the answer shows that no training athlete resembles it.

__Step 22:__ Read the five columns with the largest `feature_importances_` in the default tree.

In [ ]:
importance = pd.Series(default_fitted.feature_importances_, index=prepared_names).nlargest(5)
importance.round(4)

**`Train bf competition` carries 0.2999 of the default tree's total impurity reduction, `Cardiovascular training` 0.1888 and `Strength training` 0.1112.** These numbers say how much each column lowered impurity in this tree on these training athletes. They do not say what changes an athlete's outcome, and a column correlated with a more useful one may carry less than its share.

# <font color='#E8800A'>The settings of a tree</font> <a class="anchor" id="settings"></a>
[Back to TOC](#toc)

Eight settings of `DecisionTreeClassifier` move here, each alone with the others at their defaults: `max_depth`, `min_samples_leaf`, `min_samples_split`, `splitter`, `criterion`, `max_features`, `ccp_alpha` and `class_weight`. Each step prints training, validation and test F1 side by side, and the validation score alone picks the setting that joins the log; the tree of each logged setting is drawn beside the default's. One grid over the combinations makes the final choice, after nested cross-validation.

**`max_depth`: how many questions a path may ask.** A tree with `max_depth=d` asks at most `d` questions about an athlete before it answers, so it has at most $2^d$ leaves. `None` lets every branch grow until its athletes cannot be separated further. A shallow tree gives unlike athletes the same answer, and a deep one gives almost every training athlete an answer of its own.

**Prediction:** training F1 rises toward 1 with depth, while validation F1 rises, then falls.

__Step 23:__ Set the values of `max_depth` to try: `[2, 3, 5, 10, None]`. At `None` it is at its default.

In [ ]:
max_depth_values = [2, 3, 5, 10, None]

In the loop, the model is `DecisionTreeClassifier(max_depth=value, random_state=random_state)`, and the sweep prints training, validation and test F1 for each value.

In [ ]:
max_depth_rows = []
for value in tqdm(max_depth_values, desc='max depth values', leave=True, position=0):
    model = ...
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeClassifier(max_depth={value!r})", "setting": str(value),
           "train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    max_depth_rows.append(row)
max_depth_sweep = pd.DataFrame(max_depth_rows)
max_depth_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "depth", "leaves"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = max_depth_sweep.loc[max_depth_sweep["cv_F1"].idxmax()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, ["lost", "won"]).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(max_depth_sweep["depth"], max_depth_sweep["train_F1"], marker="o", markersize=8, linewidth=2,
         color=PLOT_ORANGE, label="training (ten folds)")
plt.plot(max_depth_sweep["depth"], max_depth_sweep["cv_F1"], marker="o", markersize=8, linewidth=2,
         color=PLOT_BLUE, label="validation (ten folds)")
plt.xlabel("depth of the tree fitted on all the training rows")
plt.ylabel("F1")
plt.ylim(0, 1.05)
plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
plt.title("Training and validation F1 against tree depth")
plt.legend()
plt.show()

**Training F1 climbs from 0.7968 at depth 2 to 1.0000 with no limit, while validation F1 peaks at 0.8440 with `max_depth=5` and falls to 0.7996 for the unrestricted tree.** `DecisionTreeClassifier(max_depth=5)` joins the log, with a test F1 of 0.8507.

**`min_samples_leaf`: the smallest group that may answer.** A split is allowed only if both children keep at least this many training athletes, so a larger value makes every answer rest on more athletes and stops the tree from isolating single ones.

**Prediction:** a larger leaf size lowers training F1 and raises validation F1, until the leaves become too coarse.

__Step 24:__ Set the values of `min_samples_leaf` to try: `[1, 5, 20, 50]`. At `1` it is at its default.

In [ ]:
min_samples_leaf_values = ...

In the loop, the model is `DecisionTreeClassifier(min_samples_leaf=value, random_state=random_state)`.

In [ ]:
min_samples_leaf_rows = []
for value in tqdm(min_samples_leaf_values, desc='min samples leaf values', leave=True, position=0):
    model = DecisionTreeClassifier(min_samples_leaf=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeClassifier(min_samples_leaf={value!r})", "setting": str(value),
           "train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    min_samples_leaf_rows.append(row)
min_samples_leaf_sweep = pd.DataFrame(min_samples_leaf_rows)
min_samples_leaf_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "depth", "leaves"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = min_samples_leaf_sweep.loc[min_samples_leaf_sweep["cv_F1"].idxmax()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, ["lost", "won"]).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

**Validation F1 reaches 0.8472 at `min_samples_leaf=20`, against 0.7996 at the default of 1, and is 0.8455 at 50.** Training F1 falls from 1.0000 to 0.8755 at 20 and 0.8562 at 50, and the tree shrinks from 504 leaves to 101 and 49. `DecisionTreeClassifier(min_samples_leaf=20)` joins the log.

**`min_samples_split`: the smallest group that may ask another question.** A node with fewer training athletes than this is a leaf. It limits the parent, where `min_samples_leaf` limits the children: with `min_samples_leaf=1`, a node of 50 athletes can still split into one athlete and 49.

**Prediction:** a larger value gives a smaller tree and a lower training F1.

__Step 25:__ Set the values of `min_samples_split` to try: `[2, 10, 50]`. At `2` it is at its default.

In [ ]:
min_samples_split_values = [2, 10, 50]

In [ ]:
min_samples_split_rows = []
for value in tqdm(min_samples_split_values, desc='min samples split values', leave=True, position=0):
    model = DecisionTreeClassifier(min_samples_split=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeClassifier(min_samples_split={value!r})", "setting": str(value),
           "train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    min_samples_split_rows.append(row)
min_samples_split_sweep = pd.DataFrame(min_samples_split_rows)
min_samples_split_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "depth", "leaves"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = min_samples_split_sweep.loc[min_samples_split_sweep["cv_F1"].idxmax()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, ["lost", "won"]).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

**`min_samples_split=50` gives the best validation F1 of its sweep, 0.8324, against 0.7996 at the default of 2 and 0.7949 at 10.** The tree keeps 130 leaves, more than the 101 of `min_samples_leaf=20`, and scores below it: limiting the parent still allows small children. `DecisionTreeClassifier(min_samples_split=50)` joins the log.

**`splitter`: how the threshold is found.** `'best'` tries every threshold of every column the split may consider, as in the toy table above, and keeps the largest impurity decrease. `'random'` draws one random threshold for each of those columns and keeps the best of the draws. It searches less, so it is faster, and the tree it grows depends on `random_state` and is usually less accurate.

**Prediction:** `'random'` lowers validation F1 for a single tree.

__Step 26:__ Set the splitters to try: `["best", "random"]`. `"best"` is the default.

In [ ]:
splitter_values = ["best", "random"]

In [ ]:
splitter_rows = []
for value in tqdm(splitter_values, desc='splitter values', leave=True, position=0):
    model = DecisionTreeClassifier(splitter=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeClassifier(splitter={value!r})", "setting": str(value),
           "train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    splitter_rows.append(row)
splitter_sweep = pd.DataFrame(splitter_rows)
splitter_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "depth", "leaves"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = splitter_sweep.loc[splitter_sweep["cv_F1"].idxmax()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**`'random'` lowers validation F1 to 0.7928 against 0.7996 for `'best'`, as predicted, and its tree is larger, 734 leaves against 504.** Its test F1 is 0.7942 against 0.8041. Validation selects, so the default wins and `DecisionTreeClassifier(splitter='best')` joins the log with the default's scores, and its tree is the default tree drawn above.

**`criterion`: the impurity a split must lower.** `'gini'` is the impurity of the toy example above. `'entropy'` replaces it with

$$H = -\sum_{c} p_c \log_2 p_c,$$

where $c$ runs over the outcomes, $p_c$ is the share of outcome $c$ in the group, and $0 \log_2 0$ is taken as 0. `'log_loss'` is the same quantity under another name, so scikit-learn builds the same splits for `'entropy'` and `'log_loss'`. Gini and entropy are both 0 for a group with one outcome and largest for equal shares, so they usually rank splits alike.

**Prediction:** `'entropy'` and `'log_loss'` build the same tree.

__Step 27:__ The right child of the toy example's best split, under both criteria: the shares of its outcomes, its Gini impurity and its entropy.

In [ ]:
right_shares = toy["won"][toy[best_candidate["column"]] > best_candidate["threshold"]].value_counts(normalize=True)
print(f"shares {right_shares.round(4).to_dict()}: Gini {1 - (right_shares ** 2).sum():.4f}, entropy {-(right_shares * np.log2(right_shares)).sum():.4f}")

**The right child of the toy split holds 4 winners and 1 loser, shares 0.8 and 0.2, and has a Gini impurity of 0.3200 and an entropy of 0.7219.** The two criteria give one mixed group two different numbers, and both would be 0 for a group with a single outcome.

__Step 28:__ Set the criteria to try: `["gini", "entropy", "log_loss"]`. `"gini"` is the default.

In [ ]:
criterion_values = ["gini", "entropy", "log_loss"]

In [ ]:
criterion_rows = []
for value in tqdm(criterion_values, desc='criterion values', leave=True, position=0):
    model = DecisionTreeClassifier(criterion=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeClassifier(criterion={value!r})", "setting": str(value),
           "train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    criterion_rows.append(row)
criterion_sweep = pd.DataFrame(criterion_rows)
criterion_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "depth", "leaves"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = criterion_sweep.loc[criterion_sweep["cv_F1"].idxmax()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**`'entropy'` and `'log_loss'` build the same tree, with 462 leaves and identical scores, as predicted, and neither beats `'gini'` on the validation folds: 0.7952 against 0.7996.** The default wins, so `DecisionTreeClassifier(criterion='gini')` joins the log with the default's scores, and its tree is the default tree drawn above.

**`max_features`: how many columns a split may consider.** `None` offers every prepared column to every split. `'sqrt'` and `'log2'` offer a random subset of that many columns, drawn again at every node; if none of them gives a valid split, the search continues through the others. A tree that sees fewer columns at each node cannot always take the best one.

**Prediction:** restricting the columns lowers validation F1 for a single tree.

__Step 29:__ Set the column counts to try: `[None, "sqrt", "log2"]`. `None` is the default.

In [ ]:
max_features_values = [None, "sqrt", "log2"]

In [ ]:
max_features_rows = []
for value in tqdm(max_features_values, desc='max features values', leave=True, position=0):
    model = DecisionTreeClassifier(max_features=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeClassifier(max_features={value!r})", "setting": str(value),
           "train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    max_features_rows.append(row)
max_features_sweep = pd.DataFrame(max_features_rows)
max_features_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "depth", "leaves"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = max_features_sweep.loc[max_features_sweep["cv_F1"].idxmax()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**Offering fewer columns lowers validation F1, to 0.7760 with `'sqrt'` and 0.7926 with `'log2'`, below the default's 0.7996, and the trees grow to 783 and 912 leaves.** The default wins, so `DecisionTreeClassifier(max_features=None)` joins the log with the default's scores, and its tree is the default tree drawn above.

**`ccp_alpha`: the price of a leaf.** After the tree is grown, cost-complexity pruning removes every branch whose impurity saving is less than `ccp_alpha` for each leaf it adds; the formula is in the pruning section below. The saving is measured in the units of the impurity, so the scale of a useful value depends on the criterion.

**Prediction:** a larger `ccp_alpha` gives a smaller tree and a lower training F1, and validation F1 rises until the pruning removes branches that were useful.

__Step 30:__ Set the values of `ccp_alpha` to try: `[0.0, 0.0001, 0.001, 0.01]`. At `0.0` it is at its default.

In [ ]:
ccp_alpha_values = [0.0, 0.0001, 0.001, 0.01]

In [ ]:
ccp_alpha_rows = []
for value in tqdm(ccp_alpha_values, desc='ccp alpha values', leave=True, position=0):
    model = DecisionTreeClassifier(ccp_alpha=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeClassifier(ccp_alpha={value!r})", "setting": str(value),
           "train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    ccp_alpha_rows.append(row)
ccp_alpha_sweep = pd.DataFrame(ccp_alpha_rows)
ccp_alpha_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "depth", "leaves"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = ccp_alpha_sweep.loc[ccp_alpha_sweep["cv_F1"].idxmax()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

In [ ]:
SuperTree(best['model'], Z_train, y_train.to_numpy(), prepared_names, ["lost", "won"]).show_tree(start_depth=best['model'].get_depth(), max_samples=500)

**`ccp_alpha=0.0001` prunes nothing, leaving 504 leaves like the default, while 0.001 leaves 65 leaves and raises validation F1 to 0.8405, and 0.01 leaves 6 leaves and 0.8309.** Training F1 falls from 1.0000 to 0.9105 and 0.8373 as the pruning deepens. `DecisionTreeClassifier(ccp_alpha=0.001)` joins the log.

**`class_weight`: how much each outcome counts.** With `'balanced'`, each outcome carries the same total weight in the impurity and in the class shares of every leaf, so the less frequent outcome counts for more. `predict` then returns the class with the larger weighted share.

**Prediction:** with the positive rate printed in Step 1, `'balanced'` moves F1 little.

__Step 31:__ Set the weightings to try: `[None, "balanced"]`. `None` is the default.

In [ ]:
class_weight_values = [None, "balanced"]

In [ ]:
class_weight_rows = []
for value in tqdm(class_weight_values, desc='class weight values', leave=True, position=0):
    model = DecisionTreeClassifier(class_weight=value, random_state=random_state)
    fold_models = [clone(model).fit(fold["X_fit"], fold["y_fit"]) for fold in tree_prepared]
    fitted = clone(model).fit(Z_train, y_train)
    row = {"configuration": f"DecisionTreeClassifier(class_weight={value!r})", "setting": str(value),
           "train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                     for fold, tree in zip(tree_prepared, fold_models)],
           "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                          for fold, tree in zip(tree_prepared, fold_models)],
           "model": fitted}
    row["train_F1"], row["cv_F1"] = np.mean(row["train"]), np.mean(row["validation"])
    row["test_F1"] = f1_score(y_test, fitted.predict(Z_test))
    row["depth"], row["leaves"] = fitted.get_depth(), fitted.get_n_leaves()
    class_weight_rows.append(row)
class_weight_sweep = pd.DataFrame(class_weight_rows)
class_weight_sweep.set_index("setting")[["train_F1", "cv_F1", "test_F1", "depth", "leaves"]].round(4)

In [ ]:
# The sweep's winner joins the log.
best = class_weight_sweep.loc[class_weight_sweep["cv_F1"].idxmax()]
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": best["configuration"],
                  "train": best["train"], "validation": best["validation"], "model": best["model"]},
       Z_test, y_test, "one setting changed", metric=f1_score, label="F1")

**`'balanced'` lowers validation F1 slightly, to 0.7960 from 0.7996, and the tree grows to 546 leaves.** The default wins, so `DecisionTreeClassifier(class_weight=None)` joins the log with the default's scores, and its tree is the default tree drawn above.

# <font color='#E8800A'>Two ways to prune</font> <a class="anchor" id="prune"></a>
[Back to TOC](#toc)

**Pre-pruning** stops the tree before it grows: `max_depth` limits a path, `min_samples_split` limits the athletes a node needs before it may split, and `min_samples_leaf` limits the athletes a child may hold. **Post-pruning** grows the tree first and then removes branches. Cost-complexity pruning keeps, for a given $\alpha$, the subtree $T$ that minimises

$$R_\alpha(T) = R(T) + \alpha\,\lvert T \rvert,$$

where $R(T)$ is the sum over the leaves of $T$ of each leaf's Gini impurity weighted by its share of the training athletes, $\lvert T \rvert$ is the number of leaves, and $\alpha$ is `ccp_alpha`, the price of one leaf. A branch is removed when the impurity it saves is less than $\alpha$ for each leaf it adds. The pruning path lists the values of $\alpha$ at which branches disappear, and it is computed from the training rows alone.

__Step 32:__ Compute the pruning path of the default tree on the training athletes: the alphas at which branches would be removed.

In [ ]:
pruning_path = default_fitted.cost_complexity_pruning_path(Z_train, y_train)
path_alphas = pruning_path.ccp_alphas[:-1]
print(f"{len(path_alphas)} effective alphas, from {path_alphas.min():.6f} to {path_alphas.max():.6f}, and"
      f" {pruning_path.ccp_alphas[-1]:.6f} for the last, which leaves the root alone")

__Step 33:__ Fit a pruned tree at 40 of those alphas, and count the leaves of each.

In [ ]:
alphas = np.unique(np.quantile(path_alphas, np.linspace(0, 1, 40)))
pruned_leaves = [DecisionTreeClassifier(ccp_alpha=alpha, random_state=random_state).fit(Z_train, y_train).get_n_leaves()
                 for alpha in tqdm(alphas, desc='alphas', leave=True, position=0)]

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(alphas[1:], pruned_leaves[1:], marker="o", markersize=5, linewidth=2, color=PLOT_BLUE)
for value in ccp_alpha_values[1:]:
    plt.axvline(value, color="grey", linestyle=":", linewidth=1.5)
plt.xscale("log")
plt.xlabel("ccp_alpha (log scale); dotted: the values swept above")
plt.ylabel("leaves of the pruned tree")
plt.title("Leaves left after cost-complexity pruning")
plt.show()

**The pruned tree keeps fewer leaves as `ccp_alpha` rises, steeply at first and slowly after 0.001.** No branch disappears for `ccp_alpha` at or below the first dotted line, 0.0001, which is why that value left 504 leaves in the sweep.

__Step 34:__ The winner of each pruning sweep, beside the default: its depth, leaves, and validation and test scores.

In [ ]:
pruning_rows = [{"kind": "none", "tree": "DecisionTreeClassifier()", "depth": default_fitted.get_depth(),
                 "leaves": default_fitted.get_n_leaves(), "cv_F1": np.mean(default_cv["validation"]),
                 "test_F1": f1_score(y_test, default_fitted.predict(Z_test))}]
for kind, sweep in [("pre-pruning", max_depth_sweep), ("pre-pruning", min_samples_leaf_sweep),
                    ("pre-pruning", min_samples_split_sweep), ("post-pruning", ccp_alpha_sweep)]:
    best = sweep.loc[sweep["cv_F1"].idxmax()]
    pruning_rows.append({"kind": kind, "tree": best["configuration"], "depth": best["depth"],
                         "leaves": best["leaves"], "cv_F1": best["cv_F1"], "test_F1": best["test_F1"]})
pd.DataFrame(pruning_rows).set_index("tree").round(4)

__Step 35:__ Print the mean training and validation F1 of the `ccp_alpha` sweep, and the gap between them.

In [ ]:
pd.DataFrame({"ccp_alpha": ccp_alpha_values, "training": ccp_alpha_sweep["train_F1"],
              "validation": ccp_alpha_sweep["cv_F1"],
              "gap": ccp_alpha_sweep["train_F1"] - ccp_alpha_sweep["cv_F1"]}).set_index("ccp_alpha").round(4)

__Step 36:__ Plot the training and validation F1 of the `ccp_alpha` sweep against `ccp_alpha`, with the chosen value marked.

In [ ]:
chosen_alpha = ccp_alpha_sweep["cv_F1"].idxmax()
plt.figure(figsize=(7, 5))
plt.plot(ccp_alpha_values, ccp_alpha_sweep["train_F1"], marker="o", markersize=8, linewidth=2,
         color=PLOT_ORANGE, label="training (ten folds)", clip_on=False)
plt.plot(ccp_alpha_values, ccp_alpha_sweep["cv_F1"], marker="o", markersize=8, linewidth=2,
         color=PLOT_BLUE, label="validation (ten folds)", clip_on=False)
plt.axvline(ccp_alpha_values[chosen_alpha], color="grey", linestyle=":", linewidth=1.5, label="chosen ccp_alpha")
plt.xscale("symlog", linthresh=ccp_alpha_values[1])
plt.xlim(0, ccp_alpha_values[-1] * 1.5)
plt.ylim(0, 1.05)
plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
plt.xlabel("ccp_alpha (symlog scale)")
plt.ylabel("F1")
plt.title("Training and validation F1 against ccp_alpha")
plt.legend()
plt.show()

**Training F1 falls from 1.0000 to 0.8373 as `ccp_alpha` rises to 0.01, while validation F1 rises from 0.7996 to 0.8405 at 0.001 and slips to 0.8309.** The two lines start far apart and end close together, and the dotted line marks the value the sweep chose.

__Step 37:__ Draw the post-pruned tree of the `ccp_alpha` sweep, every node; the default tree is drawn above.

In [ ]:
post_pruned = ccp_alpha_sweep.loc[ccp_alpha_sweep["cv_F1"].idxmax(), "model"]

In [ ]:
SuperTree(post_pruned, Z_train, y_train.to_numpy(), prepared_names, ["lost", "won"]).show_tree(start_depth=post_pruned.get_depth(), max_samples=500)

**Pre-pruning and post-pruning reach similar validation F1 with trees of different sizes:** `max_depth=5` gives 0.8440 with 29 leaves, `min_samples_leaf=20` 0.8472 with 101, `min_samples_split=50` 0.8324 with 130 and `ccp_alpha=0.001` 0.8405 with 65, against 0.7996 for the default's 504. The pruned tree has depth 12 where `max_depth=5` stops at 5: pruning after growth removes branches wherever they are not worth a leaf, at any depth.

<div class="alert alert-block alert-warning">

**The scores above describe these folds.** A setting that scores higher on the validation folds has not been shown better in general: the ten folds share athletes, so a gap between two settings is a descriptive difference and not a significance test, and following a good reason for a setting does not guarantee a better score. Here `splitter`, `criterion`, `max_features` and `class_weight` ended on or below the default, while limiting growth raised validation F1.

</div>

# <font color='#E8800A'>Nested cross-validation</font> <a class="anchor" id="nested"></a>
[Back to TOC](#toc)

The grid fits every combination of the eight settings: 256, each setting taking its default and its best other value from its own sweep. Nested cross-validation estimates how that whole grid procedure scores on athletes that took no part in its choice: the grid runs inside each of the ten outer folds on the outer training part, and the outer validation part scores its winner. It returns no model, and its mean is read beside the default's validation F1 on the same folds.

__Step 38:__ Define the grid. Each setting takes its default and its best other value from its own sweep: `max_depth: [5, None], min_samples_leaf: [1, 20], min_samples_split: [2, 50], splitter: ["best", "random"], criterion: ["gini", "entropy"], max_features: [None, "log2"], ccp_alpha: [0.0, 0.001], class_weight: [None, "balanced"]`. Type each list as written.

In [ ]:
tree_grid = [{
    "estimator__max_depth": ..., "estimator__min_samples_leaf": ...,
    "estimator__min_samples_split": ..., "estimator__splitter": ...,
    "estimator__criterion": ...,
    "estimator__max_features": ..., "estimator__ccp_alpha": ...,
    "estimator__class_weight": ...}]
tree_template = CachedSearchCV(
    PreparedEstimator(FoldPreprocessor(classification_recipe), DecisionTreeClassifier(random_state=random_state)),
    tree_grid, scoring="f1", cv=tree_folds, n_jobs=4,
)
print(f"{len(ParameterGrid(tree_grid))} combinations, each scored on {tree_folds.get_n_splits()} inner folds")

**256 combinations on ten inner folds is 2,560 fits for one grid, so nested cross-validation fits 25,600 candidates and ten winners, and the final grid 2,560 and one.** Candidates fitted on the same rows share their fold preprocessing, and four workers fit the folds in parallel.

__Step 39:__ Estimate the grid with nested cross-validation.

In [ ]:
tree_nested_fold_results = [cross_validate(tree_template, X_train, y_train, cv=[_outer_split], scoring='f1', return_train_score=True)
    for _outer_split in tqdm(list(tree_folds.split(X_train, y_train)), total=tree_folds.get_n_splits(X_train, y_train), desc='Nested CV: outer folds', unit='fold', leave=True, position=0)]
tree_nested = {key: np.concatenate([np.asarray(result[key], dtype=object if key == 'estimator' else None) for result in tree_nested_fold_results]) for key in tree_nested_fold_results[0]}
nested_validation = tree_nested["test_score"]
print(f"outer training F1 {tree_nested['train_score'].mean():.4f}; "
      f"outer validation F1 {nested_validation.mean():.4f}, fold SD {nested_validation.std(ddof=1):.4f}")

**Nested validation F1 is 0.8475 over the ten outer folds, with a fold standard deviation of 0.0186.** The outer training mean is 0.8679, close to it, because the winners of the grid are limited trees. The folds share athletes, so 0.0186 is the spread of ten overlapping scores and not a standard error.

__Step 40:__ Set the default's validation score on the same folds beside the nested estimate, and compare their paired fold scores.

In [ ]:
paired_gain = nested_validation - np.array(default_cv["validation"])
print(f"the default on the same folds: F1 {np.mean(default_cv['validation']):.4f}")
print(f"nested minus default: mean {paired_gain.mean():+.4f};"
      f" fold range {paired_gain.min():+.4f} to {paired_gain.max():+.4f};"
      f" grid higher in {(paired_gain > 0).sum()} of {len(paired_gain)} folds")

**The nested mean is 0.0479 above the default's 0.7996, and the grid is higher in all ten folds, by 0.0232 to 0.0804.** This is a descriptive comparison of overlapping splits, not a significance test.

# <font color='#E8800A'>The grid search</font> <a class="anchor" id="search"></a>
[Back to TOC](#toc)

The grid now runs once on all the training athletes, and its winner is the model this week deploys.

__Step 41:__ Run the grid on all the training athletes.

In [ ]:
tree_search = clone(tree_template).fit(X_train, y_train)

__Step 42:__ List the five best combinations, each beside the test score of its refit.

In [ ]:
results = tree_search.cv_results_
ranking = pd.DataFrame({
    "settings": [", ".join(f"{name.split('__')[-1]}={value}" for name, value in candidate.items())
                 for candidate in results["params"]],
    "cv_F1": results["mean_test_score"],
}).nlargest(5, "cv_F1")
# Each candidate is refitted on the training rows, and only then scored on the test rows.
ranking["test_F1"] = ranking.index.map(lambda index: f1_score(
    y_test, clone(tree_search.estimator).set_params(
        **clone(results["params"][index], safe=False)).fit(X_train, y_train).predict(X_test)))
print(f"{len(results['params'])} combinations; the five best:")
print(ranking.round(4).to_string())

**All five best combinations use `min_samples_leaf=20`, `criterion='gini'`, `max_features=None` and `class_weight=None`, at validation F1 0.8461 to 0.8543, and four of them grow without a depth limit.** The winner has `ccp_alpha=0.001` and `min_samples_split=50`, and scores 0.8402 on the test rows.

__Step 43:__ Log the winner's own score: its validation score on the same folds.

In [ ]:
# The winner's own score is the best of all of them on the same folds: its settings
# are fitted on each fold's prepared training part.
chosen_tree = tree_search.best_estimator_
fold_models = [clone(chosen_tree.estimator).fit(fold["X_fit"], fold["y_fit"]) for fold in tqdm(tree_prepared, desc='tree prepared', leave=True, position=0)]
winner_cv = {"train": [f1_score(fold["y_fit"], tree.predict(fold["X_fit"]))
                       for fold, tree in zip(tree_prepared, fold_models)],
             "validation": [f1_score(fold["y_score"], tree.predict(fold["X_score"]))
                            for fold, tree in zip(tree_prepared, fold_models)]}
# The log names the winner by its settings.
tree_winner_name = f"DecisionTreeClassifier({ranking.loc[tree_search.best_index_, 'settings']})"
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(), "name": tree_winner_name,
                  "train": winner_cv["train"], "validation": winner_cv["validation"],
                  "model": chosen_tree},
       X_test, y_test, "winner's own score", metric=f1_score, label="F1")

__Step 44:__ Log the winner's nested estimate, beside the same deployed model's test score.

In [ ]:
record(tree_log, {"frame": type(tree_folds).__name__, "rows": len(y_train),
                  "folds": tree_folds.get_n_splits(),
                  "name": f"joint search over {len(tree_search.cv_results_['params'])} combinations",
                  "train": tree_nested["train_score"], "validation": nested_validation,
                  "model": chosen_tree},
       X_test, y_test, "nested estimate", metric=f1_score, label="F1")

**The winner's own validation F1, 0.8543, sits above its nested estimate, 0.8475,** because it was chosen as the best of 256 on the same folds. The board publishes the nested estimate.

__Step 45:__ Draw the grid winner's tree, every node, in the zoomable viewer.

In [ ]:
SuperTree(chosen_tree.estimator_, Z_train, y_train.to_numpy(), prepared_names, ["lost", "won"]).show_tree(start_depth=chosen_tree.estimator_.get_depth(), max_samples=500)

**The grid winner has 9 levels and 38 leaves, against the default's 21 and 504, and its top question is the default's, about `Train bf competition`.** It has no `max_depth`, and `min_samples_leaf=20` and `ccp_alpha=0.001` still keep it that small.

# <font color='#E8800A'>The number we stand behind</font> <a class="anchor" id="board"></a>
[Back to TOC](#toc)

The grid's winner was chosen on the training folds. The closing comparison reports its test score beside the default, and the board publishes that score alongside the nested estimate.

__Step 46:__ Report the winner and the default on the test athletes.

In [ ]:
tuned_f1 = f1_score(y_test, chosen_tree.predict(X_test))
default_f1 = f1_score(y_test, default_fitted.predict(Z_test))
print(f"winner test F1: {tuned_f1:.4f}")
print(f"default test F1: {default_f1:.4f}")

**On the 800 held-out athletes the winner scores F1 0.8402 and the default 0.8041,** against 0.8475 nested and 0.7996 on the folds. The configuration was chosen with validation scores alone; the test scores printed during the sweeps did not select it.

__Step 47:__ Read three test metrics for the default and the winner: F1, ROC-AUC and balanced accuracy.

In [ ]:
print(f"{'':10s}{'F1':>9s}{'ROC-AUC':>10s}{'balanced acc':>14s}")
for name, model, rows in (("default", default_fitted, Z_test), ("tuned", chosen_tree, X_test)):
    predicted = model.predict(rows)
    print(f"{name:10s}{f1_score(y_test, predicted):9.4f}"
          f"{roc_auc_score(y_test, model.predict_proba(rows)[:, 1]):10.4f}"
          f"{balanced_accuracy_score(y_test, predicted):14.4f}")

**The winner is ahead of the default in all three metrics: F1 0.8402 against 0.8041, ROC-AUC 0.8518 against 0.7481, and balanced accuracy 0.7695 against 0.7481.** The default's ROC-AUC equals its balanced accuracy because its probabilities are all 0 or 1, so its ROC curve has a single corner.

__Step 48:__ Take the default's cross-validated score, on the same folds, as the baseline.

In [ ]:
# The default was measured on these same folds, so it is the baseline.
default_cv_mean = np.mean(default_cv["validation"])
print(f"default over the outer folds: F1 {default_cv_mean:.4f}")

__Step 49:__ Build the course scoreboard with the results earlier weeks saved and the floor.

In [ ]:
# The board: a result an earlier week saved joins it only if it was scored on the
# same data, recipe, rows and folds; the floor labels every athlete with the
# majority outcome.
results_dir = "../../results"
tree_board = Scoreboard("champions outcome", "F1", context=result_context(
    "../../data/interim/champions.csv", classification_recipe_log,
    y_train.index.tolist(), y_test.index.tolist(),
    outer_folds=outer_folds(tree_folds, X_train, y_train)),
    floor=round(f1_score(y_test, DummyClassifier(strategy="most_frequent").fit(
        X_train, y_train).predict(X_test)), 4))
for week, model, cv, test in results_before(9, "classification", results_dir,
                                             metric=tree_board.metric,
                                             context=tree_board.context):
    tree_board.record(week, model, cv, test)

__Step 50:__ Record this week's result on the board, save it with the fitted winner, and print the board.

In [ ]:
# The row carries two numbers: the nested cross-validated mean first, then the
# winner's held-out score. Its label names every setting the grid chose, from best_params_.
chosen_label = "DecisionTreeClassifier(" + ", ".join(
    f"{name.split('__')[-1]}={value!r}" for name, value in tree_search.best_params_.items()) + ")"
tree_board.record(9, chosen_label, round(nested_validation.mean(), 4), round(tuned_f1, 4))
tree_board.save(
    results_dir, 9, "classification", baseline_cv=round(default_cv_mean, 4),
    baseline_test=round(default_f1, 4),
)

# The fitted winner is saved beside its result, so a later week can reload it if
# it ends the course as the best model. Loading needs preprocessing.py on the path.
joblib.dump(chosen_tree, f"{results_dir}/week_09_classification_model.joblib")

print(tree_board.table())

<div class="alert alert-block alert-success">

**`DecisionTreeClassifier(ccp_alpha=0.001, min_samples_leaf=20, min_samples_split=50)`, grown without a depth limit, joins the board at F1 0.8475 over the ten outer folds and 0.8402 held out,** above the floor's 0.7529 and below the logistic regression's 0.8486 and 0.8512 and KNN's 0.8518 and 0.8615. The row and the fitted winner are saved in `results/`.

</div>

# <font color='#E8800A'>Key takeaways</font> <a class="anchor" id="takeaways"></a>
[Back to TOC](#toc)

1. **A split is chosen by the impurity reduction of its children:** on the toy athletes the best question, `hours <= 9.0`, lowers the weighted Gini impurity from 0.5000 to 0.2000.
2. **A prediction is the class share of one leaf:** the first test athlete reaches a leaf of 1,094 training athletes and a probability of 0.6773, while the unrestricted tree's leaves hold a median of 2 athletes.
3. **Limiting growth beat the default on validation:** F1 0.7996 unrestricted, against 0.8440 at `max_depth=5`, 0.8472 at `min_samples_leaf=20` and 0.8405 at `ccp_alpha=0.001`.
4. **Nested cross-validation put the grid at 0.8475, above the default's 0.7996 in all ten folds,** and the grid's winner is the model scored on the test rows.

Exit ticket: the unrestricted tree scores F1 1.0000 on the folds it was fitted on and 0.7996 on the validation folds, while the tree limited to `max_depth=5` scores 0.8643 and 0.8440. In one sentence, why does the shallower tree score higher on athletes it was not fitted on?

In [ ]:
# Write your answer below. No Python is required.
#


# <font color='#E8800A'>References</font> <a class="anchor" id="references"></a>
[Back to TOC](#toc)

- Breiman, L., Friedman, J., Olshen, R. & Stone, C. (1984). *Classification and Regression Trees*. Wadsworth.
- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning*, 2nd ed. Springer, chapter 9.2 (tree-based methods).
- James, G., Witten, D., Hastie, T. & Tibshirani, R. *An Introduction to Statistical Learning*, chapter 8.1 (the basics of decision trees).
- scikit-learn User Guide, [Decision Trees](https://scikit-learn.org/stable/modules/tree.html) · [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) · [Understanding the decision tree structure](https://scikit-learn.org/stable/auto_examples/tree/plot_unveil_tree_structure.html).